# Notebook 01 — Voting Ensembles, Condorcet's Theorem, and Model Diversity

**Session:** Ensembling  
**Module:** 02 · Calculus and Linear Algebra

## Goal
Demonstrate Condorcet's Jury Theorem via analytical binomial derivations and Monte Carlo simulations, contrast Hard Voting vs. Soft Voting on nonlinear decision boundaries, and evaluate model diversity through pairwise prediction correlation metrics (Pearson, Spearman, and Kendall).

> **Core Principle:** Combining diverse models whose individual accuracies exceed 50% ($p > 0.5$) guarantees an ensemble error rate strictly lower than any individual model, provided their errors are sufficiently uncorrelated.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

## 1. Condorcet's Jury Theorem: Analytical Formula & Monte Carlo Simulation

Consider $M$ independent binary classifiers each with individual accuracy $p = 0.70$.
Under majority voting with odd $M$, the ensemble is correct if at least $\lceil M/2 \rceil$ models are correct:

$$P_{\text{ens}}(M, p) = \sum_{k=\lceil M/2 \rceil}^M \binom{M}{k} p^k (1-p)^{M-k}$$

In [ ]:
def condorcet_analytical(num_models: int, p: float) -> float:
    """Compute exact majority voting accuracy for M independent classifiers with accuracy p."""
    k_min = (num_models // 2) + 1
    return sum(
        math.comb(num_models, k) * (p**k) * ((1 - p) ** (num_models - k))
        for k in range(k_min, num_models + 1)
    )


# Compare M = 1, 3, 5, ..., 25 for p = 0.70
m_values = list(range(1, 31, 2))
p_base = 0.70
analytical_accuracies = [condorcet_analytical(m, p_base) for m in m_values]

for m, acc in zip(m_values[:6], analytical_accuracies[:6], strict=True):
    print(f"Ensemble Size M = {m:2d} | Majority Vote Accuracy = {acc:.4f} (Base = {p_base:.2f})")

### 1.1 Monte Carlo Simulation of Voter Error Reduction
We simulate $N = 10,000$ independent classification trials for $M \in [1, 29]$ with individual probability $p = 0.70$.

In [ ]:
np.random.seed(42)
n_trials = 10000
simulated_accuracies = []

for m in m_values:
    # Shape: (n_trials, m) with 1 = correct, 0 = incorrect
    votes = np.random.binomial(n=1, p=p_base, size=(n_trials, m))
    majority_vote = (votes.sum(axis=1) >= (m // 2 + 1)).astype(int)
    simulated_accuracies.append(majority_vote.mean())

comparison_df = pd.DataFrame(
    {
        "Ensemble_Size_M": m_values,
        "Analytical_P": analytical_accuracies,
        "Simulated_P": simulated_accuracies,
        "Abs_Diff": np.abs(np.array(analytical_accuracies) - np.array(simulated_accuracies)),
    }
)
print(comparison_df.head(10).to_string(index=False))

## 2. Hard Voting vs. Soft Voting on a Nonlinear Dataset

- **Hard Voting:** Plurality/majority vote across discrete predicted classes: $\hat{y} = \text{mode}(\hat{y}_1, \dots, \hat{y}_M)$.
- **Soft Voting:** Argmax of average predicted class probabilities: $\hat{y} = \arg\max_c \frac{1}{M}\sum_{m=1}^M P_m(Y=c \mid x)$.

In [ ]:
# Generate synthetic dataset
X, y = make_moons(n_samples=600, noise=0.35, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Define 3 diverse base estimators
clf1 = LogisticRegression(random_state=42)
clf2 = RandomForestClassifier(n_estimators=50, random_state=42)
clf3 = SVC(probability=True, kernel="rbf", random_state=42)

# Fit individual models
clf1.fit(X_train, y_train)
clf2.fit(X_train, y_train)
clf3.fit(X_train, y_train)

acc1 = accuracy_score(y_test, clf1.predict(X_test))
acc2 = accuracy_score(y_test, clf2.predict(X_test))
acc3 = accuracy_score(y_test, clf3.predict(X_test))

# Hard Voting
hard_voting = VotingClassifier(
    estimators=[("lr", clf1), ("rf", clf2), ("svc", clf3)],
    voting="hard",
)
hard_voting.fit(X_train, y_train)
acc_hard = accuracy_score(y_test, hard_voting.predict(X_test))

# Soft Voting
soft_voting = VotingClassifier(
    estimators=[("lr", clf1), ("rf", clf2), ("svc", clf3)],
    voting="soft",
)
soft_voting.fit(X_train, y_train)
acc_soft = accuracy_score(y_test, soft_voting.predict(X_test))

print("=== Base Estimator vs Ensemble Accuracies ===")
print(f"Logistic Regression: {acc1:.4f}")
print(f"Random Forest:       {acc2:.4f}")
print(f"Support Vector RBF:  {acc3:.4f}")
print(f"Hard Voting:         {acc_hard:.4f}")
print(f"Soft Voting:         {acc_soft:.4f}")

## 3. Quantifying Model Diversity & Prediction Correlation

Using the sample prediction files in `../data/`, we measure pairwise correlation across model predictions:
1. **Pearson Correlation ($r$):** Linear association.
2. **Spearman Rank Correlation ($\rho$):** Monotonic rank-order association.
3. **Kendall's Tau ($\tau$):** Concordance of pairwise ranking orders.

In [ ]:
data_path = Path("../data")
file1 = data_path / "method1.csv"
file2 = data_path / "method2.csv"
file3 = data_path / "method3.csv"

df1 = pd.read_csv(file1, index_col="ImageId").rename(columns={"Label": "Method1"})
df2 = pd.read_csv(file2, index_col="ImageId").rename(columns={"Label": "Method2"})
df3 = pd.read_csv(file3, index_col="ImageId").rename(columns={"Label": "Method3"})

merged_df = df1.join(df2).join(df3).sort_index()
print("Aligned Predictions:")
print(merged_df)

print("\n=== Pearson Correlation Matrix ===")
print(merged_df.corr(method="pearson").round(4))

print("\n=== Spearman Rank Correlation Matrix ===")
print(merged_df.corr(method="spearman").round(4))

print("\n=== Kendall's Tau Correlation Matrix ===")
print(merged_df.corr(method="kendall").round(4))

# Compute pairwise disagreement rate
pairs = [("Method1", "Method2"), ("Method1", "Method3"), ("Method2", "Method3")]
print("\n=== Pairwise Disagreement Rate ===")
for col_a, col_b in pairs:
    disagree = (merged_df[col_a] != merged_df[col_b]).mean()
    print(f"Disagreement({col_a}, {col_b}): {disagree * 100:.1f}%")